# solving fits in folder

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name=version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

### 모듈 버전 확인

아래 셀을 실행하면 이 노트북을 실행한 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [1]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        #print(f"**** module {pkg} is not installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")

%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module photutils is installed
**** module ccdproc is installed
**** module version_information is installed
This notebook was generated at 2023-09-19 10:24:54 (KST = GMT+0900) 
0 Python     3.11.5 64bit [GCC 12.3.0]
1 IPython    8.15.0
2 OS         Linux 5.15.0 82 generic x86_64 with glibc2.31
3 numpy      1.25.2
4 pandas     2.1.0
5 matplotlib 3.8.0
6 scipy      1.11.2
7 astropy    5.2.1
8 photutils  1.6.0
9 ccdproc    2.4.1
10 version_information 1.0.4


### import modules

In [2]:
from glob import glob
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.stats import sigma_clip
from ccdproc import combine, ccd_process, CCDData

import ysfitsutilpy as yfu
import ysphotutilpy as ypu

import _astro_utilities
import _Python_utilities

plt.rcParams.update({'figure.max_open_warning': 0})

/home/guitar79/anaconda3/envs/astro_Python_env/lib/python3.11/site-packages/ysphotutilpy/seputil.py:113: UserWarning: Package sep is not installed. Some functions will not work.
  warn("Package sep is not installed. Some functions will not work.")


In [3]:
#%%
#######################################################
# read all files in base directory for processing
BASEDIR = Path("/mnt/Rdata/OBS_data") 

DOINGDIR = Path(BASEDIR / "ccd_test_folder")
DOINGDIR = Path('/mnt/Rdata/OBS_data/CCD_obs_raw/')

DOINGDIRs = sorted(_Python_utilities.getFullnameListOfallsubDirs(DOINGDIR))

print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))
#######################################################

len(DOINGDIRs):  106
len(DOINGDIRs):  106


In [6]:
remove = 'BIAS'
DOINGDIRs = [x for x in DOINGDIRs if remove not in x]
remove = 'DARK'
DOINGDIRs = [x for x in DOINGDIRs if remove not in x]
remove = 'FLAT'
DOINGDIRs = [x for x in DOINGDIRs if remove not in x]

#print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))
#######################################################

len(DOINGDIRs):  32


In [14]:
for DOINGDIR in DOINGDIRs[1:2] :
    DOINGDIR = Path(DOINGDIR)
    print("DOINGDIR", DOINGDIR)
    fits_in_dir = sorted(list(DOINGDIR.glob('*.fit*')))
    #print("fits_in_dir", fits_in_dir)
    print("len(fits_in_dir)", len(fits_in_dir))

    if len(fits_in_dir) == 0 :
        print(f"There is no fits fils in {DOINGDIR}")
        pass
    else : 
        print(f"Starting: {str(DOINGDIR.parts[-1])}")
    
        # MASTERDIR = DOINGDIR / _astro_utilities.master_dir
        # REDUCEDDIR = DOINGDIR / _astro_utilities.reduced_dir
        SOLVEDDIR = DOINGDIR / _astro_utilities.solved_dir2

        if not SOLVEDDIR.exists():
            os.makedirs("{}".format(str(SOLVEDDIR)))
            print("{} is created...".format(str(SOLVEDDIR)))

        summary = yfu.make_summary(DOINGDIR/"*.fit*")
        print("len(summary):", len(summary))
        print("summary:", summary)
        #print(summary["file"][0])

DOINGDIR /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin
len(fits_in_dir) 90
Starting: M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin
/mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/solved2 is created...
All 45 keywords (guessed from /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/M 13_LIGHT_b_2023-08-25-11-16-07_30sec_RiLA600_STX-16803_-20c_1bin.fit) will be loaded.
len(summary): 90
summary:                                                  file  filesize  SIMPLE  \
0   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
1   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
2   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
3   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
4   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
..                          

In [15]:
summary = yfu.make_summary(DOINGDIR/"*.fit*")
print("len(summary):", len(summary))
print("summary:", summary)
#print(summary["file"][0])

All 45 keywords (guessed from /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/M 13_LIGHT_b_2023-08-25-11-16-07_30sec_RiLA600_STX-16803_-20c_1bin.fit) will be loaded.
len(summary): 90
summary:                                                  file  filesize  SIMPLE  \
0   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
1   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
2   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
3   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
4   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
..                                                ...       ...     ...   
85  /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
86  /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
87  /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
88  /mnt/Rd

In [16]:
df_light = summary.loc[summary["IMAGETYP"] == "LIGHT"].copy()
df_light = df_light.reset_index(drop=True)
print("df_light:\n{}".format(df_light))

df_light:
                                                 file  filesize  SIMPLE  \
0   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
1   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
2   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
3   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
4   /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
..                                                ...       ...     ...   
85  /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
86  /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
87  /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
88  /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   
89  /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_...  33560640    True   

    BITPIX  NAXIS  NAXIS1  NAXIS2  EXTEND  BZERO IMAGETYP  ...   OBJCTRA  \
0  

In [17]:
#%%
for _, row  in df_light.iterrows():
    fpath = Path(row["file"])
    ccd = yfu.load_ccd(str(fpath))
    
    if ccd.header['PIXSCALE']:
        PIXc = ccd.header['PIXSCALE']
    else : 
        
        PIXc = _astro_utilities.calPixScale(ccd.header['FOCALLEN'], ccd.header['XPIXSZ'])
    print("PIXc", PIXc)

    solved = _astro_utilities.AstrometrySolver((str(fpath)), 
                                            str(SOLVEDDIR), 
                                            4,
                                            PIXc,)

PIXc 0.618795
downsample:  4
pixscale:  0.618795
str(SOLVEDDIR): /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/solved2
str(SOLVEDDIR/fpath.name): /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/solved2/M 13_LIGHT_b_2023-08-25-11-16-07_30sec_RiLA600_STX-16803_-20c_1bin.fit


kdtree_fits_io.c:274:kdtree_fits_read_tree: Kdtree header was not found in file /mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits
starkd.c:259:my_open: Failed to read kdtree from file "/mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits"
index.c:401:index_reload: Failed to read star kdtree from file /mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits
engine.c:196:engine_add_index: Failed to load index from path /mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits


b'Reading input file 1 of 1: "/mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/M 13_LIGHT_b_2023-08-25-11-16-07_30sec_RiLA600_STX-16803_-20c_1bin.fit"...\nExtracting sources...\nDownsampling by 4...\nsimplexy: found 272 sources.\nSolving...\nFailed to add index "/mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits".\nReading file "/mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/solved2/M 13_LIGHT_b_2023-08-25-11-16-07_30sec_RiLA600_STX-16803_-20c_1bin.axy"...\nField 1 did not solve (index index-4219.fits, field objects 1-10).\nField 1 did not solve (index index-4218.fits, field objects 1-10).\nField 1 did not solve (index index-4217.fits, field objects 1-10).\nField 1 did not solve (index index-4216.fits, field objects 1-10).\nField 1 did not solve (index index-4215.fits, field objects 1-10).\nField 1 did not solve (index index-4214.fits, field objects 1-10).\nField 1 did not solve (index index-4213.fits,

PIXc 0.618795
downsample:  4
pixscale:  0.618795
str(SOLVEDDIR): /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/solved2
str(SOLVEDDIR/fpath.name): /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/solved2/M 13_LIGHT_b_2023-08-25-11-16-49_30sec_RiLA600_STX-16803_-20c_1bin.fit


kdtree_fits_io.c:274:kdtree_fits_read_tree: Kdtree header was not found in file /mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits
starkd.c:259:my_open: Failed to read kdtree from file "/mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits"
index.c:401:index_reload: Failed to read star kdtree from file /mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits
engine.c:196:engine_add_index: Failed to load index from path /mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits


...

PIXc 0.618795
downsample:  4
pixscale:  0.618795
str(SOLVEDDIR): /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/solved2
str(SOLVEDDIR/fpath.name): /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/solved2/M 13_LIGHT_b_2023-08-25-11-19-36_30sec_RiLA600_STX-16803_-20c_1bin.fit


kdtree_fits_io.c:274:kdtree_fits_read_tree: Kdtree header was not found in file /mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits
starkd.c:259:my_open: Failed to read kdtree from file "/mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits"
index.c:401:index_reload: Failed to read star kdtree from file /mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits
engine.c:196:engine_add_index: Failed to load index from path /mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits


b'Reading input file 1 of 1: "/mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/M 13_LIGHT_b_2023-08-25-11-19-36_30sec_RiLA600_STX-16803_-20c_1bin.fit"...\nExtracting sources...\nDownsampling by 4...\nsimplexy: found 370 sources.\nSolving...\nFailed to add index "/mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits".\nReading file "/mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/solved2/M 13_LIGHT_b_2023-08-25-11-19-36_30sec_RiLA600_STX-16803_-20c_1bin.axy"...\nField 1 did not solve (index index-4219.fits, field objects 1-10).\nField 1 did not solve (index index-4218.fits, field objects 1-10).\nField 1 did not solve (index index-4217.fits, field objects 1-10).\nField 1 did not solve (index index-4216.fits, field objects 1-10).\nField 1 did not solve (index index-4215.fits, field objects 1-10).\nField 1 did not solve (index index-4214.fits, field objects 1-10).\nField 1 did not solve (index index-4213.fits,

PIXc 0.618795
downsample:  4
pixscale:  0.618795
str(SOLVEDDIR): /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/solved2
str(SOLVEDDIR/fpath.name): /mnt/Rdata/OBS_data/ccd_test_folder/M 13_LIGHT_-_2023-08-25_-_RiLA600_STX-16803_-_1bin/solved2/M 13_LIGHT_b_2023-08-25-11-20-17_30sec_RiLA600_STX-16803_-20c_1bin.fit


kdtree_fits_io.c:274:kdtree_fits_read_tree: Kdtree header was not found in file /mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits
starkd.c:259:my_open: Failed to read kdtree from file "/mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits"
index.c:401:index_reload: Failed to read star kdtree from file /mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits
engine.c:196:engine_add_index: Failed to load index from path /mnt/3TB1/CCD_obs/astrometry/data/4200/index-4200-23.fits


In [8]:
fpath.parent

NameError: name 'fpath' is not defined

In [12]:
# ccd = yfu.load_ccd(row["file"])
# dir(ccd)

['__abstractmethods__',
 '__array__',
 '__array_prepare__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_arithmetic',
 '_arithmetic_data',
 '_arithmetic_mask',
 '_arithmetic_meta',
 '_arithmetic_uncertainty',
 '_arithmetic_wcs',
 '_data',
 '_flags',
 '_handle_wcs_slicing_error',
 '_insert_in_metadata_fits_safe',
 '_mask',
 '_meta',
 '_prepare_then_do_arithmetic',
 '_psf',
 '_slice',
 '_slice_mask',
 '_slice_uncertainty',
 '_slice_wcs',
 '_uncertainty',
 '_unit',
 '_wcs',
 'add',
 'convert_unit_to',
 'copy',
 'data',
 'divide',
 'dtype',
 'flags',
 'header',
 'known_invalid_fits_unit_strings',
 'mask',
 'meta',
 'multi

In [13]:
ccd.header

SIMPLE  =                    T / C# FITS                                        
BITPIX  =                   16 /                                                
NAXIS   =                    2 / Dimensionality                                 
NAXIS1  =                 4096 /                                                
NAXIS2  =                 4096 /                                                
EXTEND  =                    T / Extensions are permitted                       
BZERO   =                32768 /                                                
IMAGETYP= 'LIGHT'              / Type of exposure                               
EXPOSURE=                 90.0 / [s] Exposure duration                          
EXPTIME =                 90.0 / [s] Exposure duration                          
DATE-LOC= '2023-09-08T00:00:19.856' / Time of observation (local)               
DATE-OBS= '2023-09-07T15:00:19.856' / Time of observation (UTC)                 
XBINNING=                   

In [18]:
if ccd.header['PIXSCALE']:
    PIXc = ccd.header['PIXSCALE']
    print("PIXc", PIXc)
   
else : 
    print("not")
    PIXc = _astro_utilities.calPixScale(ccd.header['FOCALLEN'], ccd.header['XPIXSZ'])
    print("PIXc", PIXc)

PIXc 0.618795


### Solving

In [20]:
solved = _astro_utilities.AstrometrySolver(row["file"], 
                                            str(DOINGDIR), 
                                            4,
                                            PIXc,)

downsample:  4
pixscale:  0.618795
str(SOLVEDDIR): /mnt/Rdata/OBS_data/ccd_test_folder/ATE (2023 QL7)-20230907-1951_LIGHT_-_2023-09-07_-_RiLA600_STX-16803_-_1bin
str(SOLVEDDIR/fpath.name): /mnt/Rdata/OBS_data/ccd_test_folder/ATE (2023 QL7)-20230907-1951_LIGHT_-_2023-09-07_-_RiLA600_STX-16803_-_1bin/ATE (2023 QL7)-20230907-1951_LIGHT_r_2023-09-07-15-00-19_90sec_RiLA600_STX-16803_-20c_1bin.fit
/mnt/Rdata/OBS_data/ccd_test_folder/ATE (2023 QL7)-20230907-1951_LIGHT_-_2023-09-07_-_RiLA600_STX-16803_-_1bin/ATE (2023 QL7)-20230907-1951_LIGHT_r_2023-09-07-15-00-19_90sec_RiLA600_STX-16803_-20c_1bin.fit is already exist...


In [19]:
import _astro_utilities
ccd = yfu.load_ccd(row["file"])
#solved = _astro_utilities.AstrometrySolver(row["file"], str(SOLVEDDIR), pixscale = ccd.header['PIXSCALE'])
solved = _astro_utilities.makingAstrometrySH(row["file"])
print(solved)
#solved = _astro_utilities.makingAstrometrySH(row["file"], str(SOLVEDDIR), pixscale = ccd.header['PIXSCALE'])

TypeError: makingAstrometrySH() missing 3 required positional arguments: 'solved_dir', 'downsample', and 'pixscale'

In [24]:
#%%
n = 0
for _, row  in df_light.iterrows():

    #fullname = fullnames[5]
    n += 1
    print('#'*40,
        "\n{2:.01f}%  ({0}/{1})".format(n, len(df_light), (n/len(df_light))*100))
    print ("Starting...\nfullname: {}".format(row["file"]))

    #_astro_utilities.KevinSolver(row["file"], solved_dir)
    #_astro_utilities.AstrometrySolver(df_light["file"][0], BASEDIR/solved_dir)
    solved = _astro_utilities.AstrometrySolver(row["file"], str(SOLVEDDIR))

######################################## 
5.0%  (1/20)
Starting...
fullname: /mnt/Rdata/OBS_data/RiLA600_2022/M42_Light_-_2022-10-24_-_RiLA600_STX-16803_-_1bin/M42_Light_v_2022-10-24-15-33-39_030sec_RiLA600_STX-16803_-20C_1bin.fit
Starting... 
/mnt/Rdata/OBS_data/RiLA600_2022/M42_Light_-_2022-10-24_-_RiLA600_STX-16803_-_1bin/M42_Light_v_2022-10-24-15-33-39_030sec_RiLA600_STX-16803_-20C_1bin.fit
str(SOLVEDDIR): /mnt/Rdata/OBS_data/RiLA600_2022/M42_Light_-_2022-10-24_-_RiLA600_STX-16803_-_1bin/solved
str(SOLVEDDIR/fpath.name) /mnt/Rdata/OBS_data/RiLA600_2022/M42_Light_-_2022-10-24_-_RiLA600_STX-16803_-_1bin/solved/M42_Light_v_2022-10-24-15-33-39_030sec_RiLA600_STX-16803_-20C_1bin.fit
b'Reading input file 1 of 1: "/mnt/Rdata/OBS_data/RiLA600_2022/M42_Light_-_2022-10-24_-_RiLA600_STX-16803_-_1bin/M42_Light_v_2022-10-24-15-33-39_030sec_RiLA600_STX-16803_-20C_1bin.fit"...\nFound an existing WCS header, will try to verify it.\nExtracting sources...\nsimplexy: found 580 sources.\nSolving...\nW